# Task 14 — SpatialVacuum full Lean verifier

Fresh public clone at an immutable raw SHA. CPU/high RAM, exact Lean/mathlib pins, module/core builds, live-ledger job comparison, complete oracle, and consistency judge.

In [ ]:
import datetime, hashlib, json, os, platform, re, shutil, subprocess, tempfile, time
from pathlib import Path

EXPECTED_SHA = '9dd2cdabda17c0af1baff530bb6d33c8d27f0ae5'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_LEAN_COMMIT = '00659f8e6071d7e46131ed643bf8003b99b044e9'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
WORK = Path(tempfile.mkdtemp(prefix='spatial-vacuum-full-'))
REPO = WORK / 'repo'
ARTIFACTS = WORK / 'artifacts'
ARTIFACTS.mkdir()
transcript = []

def log(message):
    text = str(message)
    transcript.append(text)
    print(text, flush=True)

def run(cmd, *, cwd=None, env=None, check=True):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    start = time.perf_counter()
    process = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - start
    log(process.stdout.rstrip())
    log(f'[exit {process.returncode}; elapsed {elapsed:.6f} s]')
    if check and process.returncode:
        raise RuntimeError(f'command failed: {shown}')
    return process, elapsed

utc_start = datetime.datetime.now(datetime.timezone.utc)
log(f'utc_start={utc_start.isoformat()}')
log(f'runtime={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').split('model name', 1)[1].splitlines()[0].lstrip('\t: '))
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0])
log('gpu=none (CPU runtime requested)')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
head = run(['git', 'rev-parse', 'HEAD'], cwd=REPO)[0].stdout.strip()
if head != EXPECTED_SHA:
    raise RuntimeError(f'HEAD mismatch: {head}')
toolchain = (REPO / 'lean-toolchain').read_text().strip()
manifest = (REPO / 'lake-manifest.json').read_text()
if toolchain != EXPECTED_TOOLCHAIN or EXPECTED_MATHLIB not in manifest:
    raise RuntimeError('toolchain or mathlib pin mismatch')
installer = WORK / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh', '-o', str(installer)])
installer_sha = hashlib.sha256(installer.read_bytes()).hexdigest()
log(f'elan_installer_sha256={installer_sha}')
env = os.environ.copy()
env['ELAN_HOME'] = str(WORK / 'elan')
env['PATH'] = str(WORK / 'elan' / 'bin') + os.pathsep + env['PATH']
env['ELAN_TOOLCHAIN'] = EXPECTED_TOOLCHAIN
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'], env=env)
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
lean_version = run(['lean', '--version'], env=env)[0].stdout
if EXPECTED_LEAN_COMMIT not in lean_version:
    raise RuntimeError('Lean commit mismatch')
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
module, module_seconds = run(['lake', 'build', 'YangMills.OS.SpatialVacuum'], cwd=REPO, env=env)
module_match = re.search(r'Build completed successfully \((\d+) jobs\)', module.stdout)
if not module_match:
    raise RuntimeError('module build did not print a measured job count')
core, core_seconds = run(['lake', 'build', 'YangMillsCore'], cwd=REPO, env=env)
core_match = re.search(r'Build completed successfully \((\d+) jobs\)', core.stdout)
if not core_match:
    raise RuntimeError('core build did not print a measured job count')
module_jobs = int(module_match.group(1))
core_jobs = int(core_match.group(1))
ledger = (REPO / 'docs' / 'VERIFICATION-LEDGER.md').read_text(encoding='utf-8')
baseline_matches = re.findall(r'The live core baseline is the latest measured ledger baseline, \*\*(\d+)\*\*', ledger)
if not baseline_matches:
    raise RuntimeError('live core baseline not found in versioned ledger')
live_baseline = int(baseline_matches[-1])
if core_jobs != live_baseline + 3:
    raise RuntimeError(f'core jobs {core_jobs} != live ledger baseline {live_baseline} + one merged-main job + two branch modules')
oracle, oracle_seconds = run(['lake', 'env', 'lean', 'oracle_check.lean'], cwd=REPO, env=env)
if 'sorryAx' in oracle.stdout:
    raise RuntimeError('sorryAx appeared in permanent oracle output')
allowed_axioms = {'propext', 'Classical.choice', 'Quot.sound'}
required = ['periodic_antiperiodic_log_difference_eq_neg_two_artanh', 'periodic_antiperiodic_log_norm_sums_lt', 'arcosh_circle_log_mixture', 'physical_arcosh_circle_log_mixture', 'circle_log_kernel_factorization', 'circle_log_kernel_eq_log_norm', 'existsUnique_circleLogKernelParameter', 'circleAverage_log_kernel_eq_log_norm']
axiom_payload_pattern = re.compile(r" depends on axioms:\s*\[([^\]]*)\]", re.DOTALL)
axiom_blocks = axiom_payload_pattern.findall(oracle.stdout)
marker_count = oracle.stdout.count(' depends on axioms:')
if len(axiom_blocks) != marker_count:
    raise RuntimeError(f'oracle axiom block parse mismatch: {len(axiom_blocks)} != {marker_count}')
required_axioms = {}
for name in required:
    marker = re.escape(f"'YangMills.OS.{name}' depends on axioms:")
    matches = re.findall(marker + r'\s*\[([^\]]*)\]', oracle.stdout, re.DOTALL)
    if len(matches) != 1:
        raise RuntimeError(f'oracle axiom block count for {name}: {len(matches)}')
    payload = matches[0]
    used = {item.strip() for item in payload.split(',') if item.strip()}
    if used != allowed_axioms:
        raise RuntimeError(f'{name} axioms {used} != {allowed_axioms}')
    required_axioms[name] = sorted(used)
for index, payload in enumerate(axiom_blocks):
    used = {item.strip() for item in payload.split(',') if item.strip()}
    if not used <= allowed_axioms:
        raise RuntimeError(f'nonstandard axioms in oracle block {index}: {used - allowed_axioms}')
consistency, consistency_seconds = run(['python3', 'scripts/check_consistency.py'], cwd=REPO, env=env)
utc_end = datetime.datetime.now(datetime.timezone.utc)
log(f'utc_end={utc_end.isoformat()}')
log(f'module_jobs_measured={module_jobs}')
log(f'core_jobs_measured={core_jobs}')
log(f'live_ledger_baseline={live_baseline}')
log(f'core_job_delta={core_jobs - live_baseline}')
log(f'module_seconds={module_seconds:.6f}')
log(f'core_seconds={core_seconds:.6f}')
log(f'oracle_seconds={oracle_seconds:.6f}')
log(f'consistency_seconds={consistency_seconds:.6f}')
log('NEW_DECLARATION_AXIOMS_EXACT=' + json.dumps(required_axioms, sort_keys=True, separators=(',', ':')))
log('SPATIAL VACUUM FULL VERIFICATION STEPS PASS')
metadata = {
    'repo_sha': head, 'toolchain': toolchain, 'lean_commit': EXPECTED_LEAN_COMMIT,
    'mathlib_pin': EXPECTED_MATHLIB, 'runtime': platform.platform(),
    'python': platform.python_version(), 'cpu_count': os.cpu_count(),
    'memory': Path('/proc/meminfo').read_text(errors='replace').splitlines()[0],
    'gpu': 'none', 'elan_installer_sha256': installer_sha,
    'utc_start': utc_start.isoformat(), 'utc_end': utc_end.isoformat(),
    'module_jobs': module_jobs, 'core_jobs': core_jobs,
    'live_ledger_baseline': live_baseline, 'core_job_delta': core_jobs - live_baseline,
    'module_seconds': module_seconds, 'core_seconds': core_seconds,
    'oracle_seconds': oracle_seconds, 'consistency_seconds': consistency_seconds,
    'required_axioms': required_axioms,
}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(ARTIFACTS / 'oracle_output.txt').write_text(oracle.stdout, encoding='utf-8')
(ARTIFACTS / 'consistency_output.txt').write_text(consistency.stdout, encoding='utf-8')
(ARTIFACTS / 'transcript.txt').write_text('\n'.join(transcript) + '\n', encoding='utf-8')
hashes = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted(ARTIFACTS.iterdir())}
(ARTIFACTS / 'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name, digest in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_vacuum_full_artifacts', 'zip', ARTIFACTS)
archive_hash = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
print(f'artifact_zip_sha256={archive_hash}', flush=True)
print('SPATIAL VACUUM FULL PASS', flush=True)
from google.colab import files
files.download(archive)
